
# 04 · Verification & invariants — Person 2 sign-off

**Vai trò notebook**: chứng minh code đảm bảo (1) no lookahead, (2) reproducibility, (3) VN market rules, (4) parse safety. Khi notebook xanh hết, Person 2 sign-off vào `.agent/sign-off-person2.md`.

**Owner**: Person 2
**Deadline**: 2026-05-23
**Slide chapter**: 6 — Conclusions (Reliability section) + appendix

## Mục tiêu
Notebook này KHÁC 3 cái trên: **mỗi cell là một `assert`**. Pass = invariant guaranteed. Fail = bug → escalate.

## Defense Q&A
- Q: Làm sao đảm bảo không có data leak?
- Q: Reproducible không? Same seed → same trajectory?
- Q: VN rules (±7%, lot-100, fees) implement đúng chưa?
- Q: Multi-agent có hallucinate JSON output không?

## Sign-off output
Cell cuối in bảng:
```
[ PASS ] No lookahead bias (env state at T only exposes timestamp < T)
[ PASS ] News D+1 shift (published news visible from next session close)
[ PASS ] Reproducibility (same seed → bit-identical metrics_table.csv)
[ PASS ] VN price band ±7% (HOSE limit clamping)
[ PASS ] VN lot-100 rounding
[ PASS ] VN asymmetric fees (buy 0.15%, sell 0.25%)
[ PASS ] LLM model lock (only gpt-4o + gpt-4o-mini)
[ PASS ] Multi-agent parse safety (0 failures over 51 decisions)
```
Nếu có FAIL → Person 2 escalate cho Duc trước khi sign-off.


## Setup


In [ ]:
import sys
from pathlib import Path

# Make `from _shared import ...` work whether you run from notebooks/ or repo root.
_NB_DIR = Path().resolve()
if _NB_DIR.name != "notebooks":
    _NB_DIR = _NB_DIR / "notebooks"
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from _shared import (  # noqa: E402
    AGENT_COLORS,
    BASELINES,
    DATA,
    FIGURES,
    LLM_AGENTS,
    RESULTS,
    RL_AGENTS,
    ROLE_COLORS,
    TRANSCRIPTS,
    assert_frozen_snapshot,
    list_transcript_dates,
    load_curve,
    load_holdings,
    load_metrics_json,
    load_metrics_table,
    load_transcript,
    save_fig,
    setup_matplotlib,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

setup_matplotlib()
assert_frozen_snapshot()
metrics = load_metrics_table()
print("snapshot OK · agents:", list(metrics.index))



## TODO-01: lookahead-env-state — chứng minh `_get_state(t)` chỉ expose `timestamp < t`
- **OWNER**: Person 2   **DEPENDS**: none
- **READ**: `src/trading_env.py:VNTradingEnv._get_state`, `tests/test_trading_env.py`
- **WRITE**: 1 `assert` cell + 1 markdown explanation
- **CONSTRAINTS**:
  - Load env với train data; tại bước t=100 (random), call `_get_state()` và check không có column nào có timestamp ≥ t.
  - Inspect feature columns: phải toàn là past prices, lagged indicators
- **VALIDATE**:
  ```python
  state = env._get_state(t=100)
  assert env.market_data.dates[100] not in state['price_window']  # current day excluded
  ```
- **PATTERN**: `tests/test_trading_env.py::test_no_lookahead_in_state`
- **DEFENSE EVIDENCE**: cell này + test file path


In [ ]:
# TODO-01: assert env state has no future data
from src.trading_env import VNTradingEnv
# ...



## TODO-02: lookahead-news-shift — news ngày D chỉ visible từ D+1 close
- **OWNER**: Person 2   **DEPENDS**: none
- **READ**: `src/data_pipeline/news_align.py:shift_news_to_dplus1`, news data
- **WRITE**: assertion + sample table
- **CONSTRAINTS**:
  ```python
  news = pd.read_parquet(DATA / 'news.parquet')
  assert (news['visible_from'] > news['published_date']).all()
  # spot-check: published Sat → visible Mon (skip weekend)
  ```
- **PATTERN**: `tests/test_news_align.py`
- **DEFENSE EVIDENCE**: assertion + spot-check row showing Sat→Mon shift


In [ ]:
# TODO-02: assert news shifted D→D+1
pass



## TODO-03: reproducibility — same seed produces bit-identical metrics
- **OWNER**: Person 2   **DEPENDS**: none
- **WRITE**: re-run `src.eval.aggregate.build_metrics_table` vs committed CSV
- **CONSTRAINTS**:
  - Load committed `results/metrics_table.csv`
  - Re-aggregate từ per-agent `metrics.json` files
  - assert numerical equality with `np.isclose(rtol=1e-9)`
  - **DO NOT** re-run backtest scripts (frozen snapshot policy)
- **VALIDATE**: bit-identical for buy_and_hold, equal_weight (deterministic). For RL/LLM, depend on seed.
- **PATTERN**: `src/eval/aggregate.py:build_metrics_table`
- **DEFENSE EVIDENCE**: assertion passes → reproducibility guaranteed at aggregation layer


In [ ]:
# TODO-03: re-aggregate vs committed CSV
from src.eval.aggregate import build_metrics_table
rebuilt = build_metrics_table(RESULTS)
# ...



## TODO-04: vn-rules-price-band — ±7% HOSE clamp
- **OWNER**: Person 2   **DEPENDS**: none
- **WRITE**: simulate transaction at hypothetical extreme price, assert clamped to band
- **CONSTRAINTS**:
  ```python
  from src.trading_env import _execute_with_vn_rules  # hoặc method name tương đương
  # construct trade with desired price = ref_price * 1.10 (10% > band)
  # assert executed_price <= ref_price * 1.07
  ```
- **PATTERN**: `tests/test_trading_env.py::test_price_band_clamp`
- **DEFENSE EVIDENCE**: assertion + reference PRD §15 + HOSE regulation


In [ ]:
# TODO-04: assert ±7% price band clamping
pass



## TODO-05: vn-rules-lot-100 — round to nearest 100 shares
- **OWNER**: Person 2   **DEPENDS**: none
- **WRITE**: assertion on rounding logic
- **CONSTRAINTS**:
  - test với order 137 shares → executed 100; order 150 → 100 or 200 depending on rounding rule
  - check `holdings.parquet` của baselines: tất cả counts %% 100 == 0
- **VALIDATE**:
  ```python
  for agent in BASELINES:
      h = load_holdings(agent)
      for t in ['VCB', 'FPT', 'HPG', 'VIC', 'VNM']:
          assert (h[t] % 100 == 0).all(), f"{agent} {t} has non-100 lot"
  ```
- **PATTERN**: `tests/test_trading_env.py::test_lot_100_rounding`


In [ ]:
# TODO-05: assert all holdings are multiples of 100
pass



## TODO-06: vn-rules-fees — asymmetric 0.15% buy / 0.25% sell
- **OWNER**: Person 2   **DEPENDS**: none
- **WRITE**: assertion on total_cost
- **CONSTRAINTS**:
  - Calculate expected fees from holdings transitions
  - Compare with metrics `total_cost` for buy_and_hold (least trades → smallest cost)
  - assert buy_and_hold total_cost > 0, < 0.1% of initial capital (because only 1 buy at t=0)
- **VALIDATE**:
  ```python
  bh_cost = metrics.loc['buy_and_hold', 'total_cost']
  assert 0 < bh_cost / 1e9 < 0.002  # < 0.2% of 1B initial
  ```
- **PATTERN**: `src/trading_env.py:_apply_fees`


In [ ]:
# TODO-06: assert fee structure plausible
pass



## TODO-07: llm-model-lock — only gpt-4o + gpt-4o-mini called
- **OWNER**: Person 2   **DEPENDS**: none
- **READ**: `src/llm/client.py:OpenAIClient.ALLOWED_MODELS`, transcripts
- **WRITE**: scan all transcripts for `model` field, assert ∈ allowed set
- **CONSTRAINTS**:
  ```python
  allowed = {"gpt-4o", "gpt-4o-mini"}
  for date in list_transcript_dates():
      t = load_transcript(date)
      for entry in t['transcript']:
          model = entry.get('model')
          if model:
              assert model in allowed, f"unexpected model {model} on {date}"
  ```
- **VALIDATE**: 51 transcripts × ~10 entries = ~500 model fields, all match
- **DEFENSE EVIDENCE**: locking proven empirically across full snapshot


In [ ]:
# TODO-07: scan transcripts for model lock compliance
pass



## TODO-08: multi-agent-parse-safety — 0 hallucination / parse failure
- **OWNER**: Person 2   **DEPENDS**: none
- **READ**: `results/multi_agent/decisions.jsonl`, transcripts
- **WRITE**: assertion + count
- **CONSTRAINTS**:
  ```python
  decisions = [json.loads(l) for l in open(RESULTS / 'multi_agent' / 'decisions.jsonl')]
  assert all(d['parse_ok'] for d in decisions)
  assert all(d['timed_out'] is False for d in decisions)
  assert all(d['node_errors_count'] == 0 for d in decisions)
  print(f"All {len(decisions)} decisions: parse OK, no timeout, no node error")
  ```
- **VALIDATE**: 51/51 clean
- **DEFENSE EVIDENCE**: hard data — 51 LLM runs, 0 failures


In [ ]:
# TODO-08: multi_agent parse safety
pass



## TODO-09: sign-off-table — render final pass/fail table

- **OWNER**: Person 2   **DEPENDS**: TODO-01..08
- **WRITE**: code cell prints sign-off table; markdown cell with Person 2's signature
- **CONSTRAINTS**:
  - Use a `RESULTS_DICT: dict[str, bool]` accumulated as TODO-01..08 run
  - Print formatted table:
    ```
    ┌───────────────────────────────────────────┬──────┐
    │ Invariant                                 │ Status │
    ├───────────────────────────────────────────┼──────┤
    │ No lookahead — env state                  │ PASS │
    │ No lookahead — news D+1                   │ PASS │
    │ Reproducibility — bit-identical aggregate │ PASS │
    │ VN ±7% price band                         │ PASS │
    │ VN lot-100 rounding                       │ PASS │
    │ VN fees asymmetric                        │ PASS │
    │ LLM model lock                            │ PASS │
    │ Multi-agent parse safety                  │ PASS │
    └───────────────────────────────────────────┴──────┘
    ```
  - If any FAIL → cell raises RuntimeError, Person 2 doesn't sign
- **VALIDATE**: table renders + no errors


In [ ]:
# TODO-09: final sign-off table
results_dict = {}
# ...



## Defense Q&A — câu trả lời sẵn

> **Q1: Bằng chứng nào chứng minh không lookahead?**
> A: 3 layers protection: (1) env `_get_state(t)` strict timestamp < t (TODO-01). (2) News shift D→D+1 hardcoded ở data pipeline (TODO-02). (3) Unit tests in `tests/test_trading_env.py` + `tests/test_news_align.py` chạy mỗi PR.

> **Q2: Reproducible?**
> A: Có. Same seed → same trajectory cho deterministic agents (baselines, RL với seed). LLM cached bằng `(date, ticker_set, prompt_hash)` → rerun free. Aggregation bit-identical (TODO-03).

> **Q3: VN rules đầy đủ?**
> A: ±7% price band, lot-100, asymmetric fees (TODO-04, 05, 06). T+2 settlement không implement (out of scope, PRD §4 nice-to-have).

> **Q4: Multi-agent có hallucinate JSON?**
> A: 0 parse failures qua 51 decisions (TODO-08). Robust nhờ portfolio_manager output wrapped trong markdown JSON fence + parser fallback to hold action.

## Sign-off

Khi tất cả assertion PASS:

```markdown
# Person 2 sign-off — invariants verification

Date: 2026-05-XX
Notebook: notebooks/04_invariants.ipynb (commit <sha>)

Tất cả 8 invariants pass. Approved for thesis defense.

Signed: <tên Person 2>
```

Lưu file đó vào `.agent/sign-off-person2.md`.
